# CF vs Log-Likelihood Gap Validation

This notebook reproduces the validation study demonstrating that CF (pre-inference) predicts inference quality metrics (post-inference) for the Stochastic Volatility Filter model.

**Key findings:**
- CF strongly predicts log-likelihood gap (r = 0.86)
- CF moderately predicts MSE ratio (r = 0.40)
- PSIS-k̂ is not appropriate for Gaussian posteriors

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from dataclasses import dataclass
from typing import Tuple, NamedTuple
import warnings

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13

np.random.seed(42)

## 1. SVF Model Implementation

In [ ]:
@dataclass
class SVFParams:
    """SVF model parameters."""
    coupling: float = 0.5
    base_volatility: float = 0.5
    volatility_noise: float = 0.3
    observation_noise: float = 0.5


class SVFSimulation(NamedTuple):
    x3: np.ndarray    # Volatility driver
    x2: np.ndarray    # State
    y: np.ndarray     # Observations
    vol: np.ndarray   # Instantaneous volatility
    params: SVFParams


def simulate_svf(params: SVFParams, T: int = 300, seed: int = None) -> SVFSimulation:
    """Simulate from SVF generative model."""
    if seed is not None:
        np.random.seed(seed)
    
    x3 = np.zeros(T)
    x2 = np.zeros(T)
    vol = np.zeros(T)
    y = np.zeros(T)
    
    vol[0] = params.base_volatility
    y[0] = np.random.normal(0, params.observation_noise)
    
    for t in range(1, T):
        x3[t] = x3[t-1] + np.random.normal(0, params.volatility_noise)
        log_vol = np.clip(params.coupling * x3[t], -3, 3)
        vol[t] = np.clip(params.base_volatility * np.exp(log_vol), 0.1, 5.0)
        x2[t] = x2[t-1] + np.random.normal(0, vol[t])
        y[t] = x2[t] + np.random.normal(0, params.observation_noise)
    
    return SVFSimulation(x3=x3, x2=x2, y=y, vol=vol, params=params)

## 2. Kalman Filter Implementation

In [ ]:
class KalmanResult(NamedTuple):
    x_filtered: np.ndarray
    P_filtered: np.ndarray
    log_likelihood: float


def kalman_filter(y: np.ndarray, process_var: np.ndarray, obs_var: float) -> KalmanResult:
    """Kalman filter for local level model."""
    T = len(y)
    if np.isscalar(process_var):
        process_var = np.full(T, process_var)
    
    x_filt = np.zeros(T)
    P_filt = np.zeros(T)
    P_filt[0] = 1.0
    log_lik = 0.0
    
    for t in range(1, T):
        # Predict
        x_pred = x_filt[t-1]
        P_pred = P_filt[t-1] + process_var[t]
        
        # Update
        S = P_pred + obs_var
        K = P_pred / S
        innovation = y[t] - x_pred
        
        x_filt[t] = x_pred + K * innovation
        P_filt[t] = (1 - K) * P_pred
        
        log_lik += -0.5 * (np.log(2 * np.pi * S) + innovation**2 / S)
    
    return KalmanResult(x_filt, P_filt, log_lik)


def fit_mfvi(sim: SVFSimulation) -> Tuple[KalmanResult, float]:
    """Fit MFVI (constant volatility) to SVF data."""
    obs_var = sim.params.observation_noise ** 2
    
    def neg_ll(log_sigma):
        sigma = np.exp(float(log_sigma))
        return -kalman_filter(sim.y, sigma**2, obs_var).log_likelihood
    
    # Grid search + refinement
    best = (sim.params.base_volatility, neg_ll(np.log(sim.params.base_volatility)))
    for s in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
        ll = neg_ll(np.log(s))
        if ll < best[1]:
            best = (s, ll)
    
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            res = minimize(neg_ll, np.log(best[0]), method='Nelder-Mead')
            sigma_mf = np.exp(float(np.atleast_1d(res.x)[0]))
    except:
        sigma_mf = best[0]
    
    return kalman_filter(sim.y, sigma_mf**2, obs_var), sigma_mf


def fit_oracle(sim: SVFSimulation) -> KalmanResult:
    """Fit oracle (true volatility) filter."""
    return kalman_filter(sim.y, sim.vol**2, sim.params.observation_noise**2)

## 3. CF Computation

In [ ]:
def compute_cf_svf(sim: SVFSimulation) -> float:
    """Compute CF for SVF measuring volatility-state coupling."""
    x3 = sim.x3[1:]
    dx2 = np.diff(sim.x2)
    log_abs_dx2 = np.log(np.abs(dx2) + 1e-10)
    
    rho = np.corrcoef(x3, log_abs_dx2)[0, 1]
    if not np.isfinite(rho):
        return np.nan
    
    # MI for Gaussian
    rho_clipped = np.clip(rho, -0.9999, 0.9999)
    mi = -0.5 * np.log(1 - rho_clipped**2)
    
    # Entropies
    sigma_z = max(np.std(x3), 1.0)
    sigma_x = max(np.std(log_abs_dx2), 1.0)
    h_z = 0.5 * np.log(2 * np.pi * np.e * sigma_z**2)
    h_x = 0.5 * np.log(2 * np.pi * np.e * sigma_x**2)
    h_min = min(h_z, h_x)
    
    return np.clip(mi / h_min, 0.0, 1.0) if h_min > 0 else np.nan

## 4. Run Validation Study

In [ ]:
# Configuration
coupling_values = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
n_sims = 100
T = 300

results = []
for kappa in coupling_values:
    print(f'Processing κ = {kappa}...', end=' ')
    for rep in range(n_sims):
        params = SVFParams(coupling=kappa)
        sim = simulate_svf(params, T=T, seed=int(kappa*10000+rep))
        
        cf = compute_cf_svf(sim)
        kf_mfvi, _ = fit_mfvi(sim)
        kf_oracle = fit_oracle(sim)
        
        mse_mf = np.mean((kf_mfvi.x_filtered - sim.x2)**2)
        mse_oracle = np.mean((kf_oracle.x_filtered - sim.x2)**2)
        mse_ratio = mse_mf / max(mse_oracle, 1e-10)
        
        ll_gap = kf_oracle.log_likelihood - kf_mfvi.log_likelihood
        
        results.append({
            'coupling': kappa,
            'cf': cf,
            'mse_ratio': mse_ratio,
            'log_lik_gap': ll_gap
        })
    print('done')

df = pd.DataFrame(results)
print(f'\nTotal simulations: {len(df)}')

## 5. Results Summary

In [ ]:
summary = df.groupby('coupling').agg({
    'cf': ['mean', 'std'],
    'mse_ratio': ['mean', 'std'],
    'log_lik_gap': ['mean', 'std']
}).round(3)

print('Results by Coupling Strength:')
print(summary)

In [ ]:
# Correlation analysis
valid = df.dropna()
print('\nCorrelation Analysis:')
print('-' * 50)

r_cf_ll, p = stats.pearsonr(valid['cf'], valid['log_lik_gap'])
print(f'CF vs Log-Lik Gap: r = {r_cf_ll:.3f}, p = {p:.2e}')

r_cf_mse, p = stats.pearsonr(valid['cf'], valid['mse_ratio'])
print(f'CF vs MSE Ratio:   r = {r_cf_mse:.3f}, p = {p:.2e}')

r_ll_mse, p = stats.pearsonr(valid['log_lik_gap'], valid['mse_ratio'])
print(f'ΔLL vs MSE Ratio:  r = {r_ll_mse:.3f}, p = {p:.2e}')

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Panel A: CF vs Log-Lik Gap
ax = axes[0]
scatter = ax.scatter(valid['cf'], valid['log_lik_gap'], 
                     c=valid['coupling'], cmap='viridis',
                     alpha=0.6, s=25)
slope, intercept, r, p, se = stats.linregress(valid['cf'], valid['log_lik_gap'])
x_line = np.linspace(0, valid['cf'].max() * 1.1, 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', lw=2.5)
ax.set_xlabel('CF (pre-inference)')
ax.set_ylabel('Log-Likelihood Gap')
ax.set_title(f'(A) CF predicts inference quality\n(r = {r:.2f})')
plt.colorbar(scatter, ax=ax, label='Coupling κ')

# Panel B: Summary by coupling
ax = axes[1]
kappas = summary.index.values
ll_means = summary[('log_lik_gap', 'mean')].values
ll_stds = summary[('log_lik_gap', 'std')].values
mse_means = summary[('mse_ratio', 'mean')].values
mse_stds = summary[('mse_ratio', 'std')].values

ax.errorbar(kappas, ll_means, yerr=ll_stds, fmt='o-', capsize=5, 
            color='blue', linewidth=2, markersize=10, label='ΔLL')
ax2 = ax.twinx()
ax2.errorbar(kappas, mse_means, yerr=mse_stds, fmt='s--', capsize=5, 
             color='red', linewidth=2, markersize=8, label='MSE')
ax.set_xlabel('Coupling κ')
ax.set_ylabel('Log-Likelihood Gap', color='blue')
ax2.set_ylabel('MSE Ratio', color='red')
ax.set_title('(B) Both metrics increase with κ')

# Panel C: Correlation bars
ax = axes[2]
metrics = ['Log-Lik Gap', 'MSE Ratio']
correlations = [r_cf_ll, r_cf_mse]
colors = ['#2ecc71', '#f39c12']
bars = ax.bar(metrics, correlations, color=colors, edgecolor='black', linewidth=1.5)
ax.axhline(0, color='black', linewidth=1)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.7)
ax.set_ylabel('Correlation with CF')
ax.set_title('(C) CF correlations')
ax.set_ylim(0, 1)
for bar, val in zip(bars, correlations):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.03, f'{val:.2f}', 
            ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_psis_validation_notebook.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 7. Classification Performance

In [ ]:
cf_threshold = 0.10
ll_threshold = valid['log_lik_gap'].median()

cf_pos = valid['cf'] > cf_threshold
ll_pos = valid['log_lik_gap'] > ll_threshold

tp = ((cf_pos) & (ll_pos)).sum()
fp = ((cf_pos) & (~ll_pos)).sum()
tn = ((~cf_pos) & (~ll_pos)).sum()
fn = ((~cf_pos) & (ll_pos)).sum()

print(f'Classification Performance (CF > {cf_threshold} → ΔLL > {ll_threshold:.1f})')
print('-' * 50)
print(f'Sensitivity: {100*tp/(tp+fn):.1f}%')
print(f'Specificity: {100*tn/(tn+fp):.1f}%')
print(f'PPV:         {100*tp/(tp+fp):.1f}%')
print(f'NPV:         {100*tn/(tn+fn):.1f}%')
print(f'\nConfusion Matrix: TP={tp}, FP={fp}, TN={tn}, FN={fn}')

## 8. Save Results

In [ ]:
df.to_csv('psis_validation_results.csv', index=False)
summary.to_csv('psis_validation_summary.csv')
print('Results saved to psis_validation_results.csv')
print('Summary saved to psis_validation_summary.csv')

## 9. Key Conclusions

1. **CF strongly predicts log-likelihood gap** (r ≈ 0.86)
   - This is the most direct measure of MFVI quality
   - Validates CF as a pre-inference diagnostic

2. **CF moderately predicts MSE ratio** (r ≈ 0.40)
   - Consistent with the manuscript's main claims

3. **PSIS-k̂ is not appropriate for this setting**
   - Both posteriors are Gaussian → light-tailed importance weights
   - Use log-likelihood gap instead for Gaussian models